# Colour Metrics over Densities - Inference

In [ ]:
try:
    import mat73
except ImportError:
    pass

from pathlib import Path
from typing import Sequence

import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

In [ ]:
path = "../../"
path = Path(path).expanduser()
import sys

sys.path.insert(0, str(path))

In [ ]:
import decode
import decode.neuralfitter.inference.functional as infer_func
print(decode.__file__)
log = decode.generic.logging.get_logger(__name__)

%config InlineBackend.figure_format='retina'

In [ ]:
# some paths and hardware settings
path_trafo = "../../calibration/FigS5-Pos0_240802_NC_BeadCal_DualColor_Z_1_MMStack_Default.ome_trafo.mat" 

path_out = "../../data/Fig1c_S5-density/FigS5-dual_color/n_frames-1000_n_steps-16_seed-42-3"
path_out = Path(path_out).expanduser()

trafo_size_ref = 512
trafo_mirr_dim = 0

device = ["cuda:0"]

path_trafo = Path(path_trafo).expanduser()

In [ ]:
path_scen_base = "../../data/Fig1c_S5-density/FigS5-dual_color/n_frames-1000_n_steps-16_seed-42-3"
path_scen_base = Path(path_scen_base)

path_scen = path_scen_base / "scenarios.pq"
scen = pd.read_parquet(path_scen)

path_frames = path_scen_base / "frames.pt"
frames = torch.load(path_frames)

img_shape = frames[0][0].shape[-2:]

scen_model = pd.DataFrame(index=scen.index)

pm = {
    # "high": "~/decode_storage/dev/lucas/training/dual_sim/2024-02-09_15-55-27-849250",
    "medium": "../../outputs/FigS5-dual_color-density-separated_ch-2026-05-14_19-24-20-015714",
    # "low": "~/decode_storage/dev/lucas/training/dual_sim/2024-02-09_17-16-13-546570",
}
pm = {k: Path(v).expanduser() for k, v in pm.items()}

# map models to scenarios
scen_model["path_model"] = None
for snr in pm:
    scen_model.loc[snr, "path_model"] = pm[snr]

scen_model

In [ ]:
# construct trafo - DO NOT EDIT - COPY FROM SAMPLE
offset = [-2, -23, 0]
trafo = decode.io.trafo.load_xyz_trafo(
    path_trafo,
    scale=1 / 1000.0,
    switch_xy=True,
    shift=(1.0, 1.0, 0.0),
    reference="trafo_inv_raw",
    device=device[0],
)
t_mirr = decode.simulation.trafo.pos.trafo.XYZMirrorAt.from_frame_flip(
    trafo_size_ref,
    trafo_mirr_dim,
    device=device[0],
)
t_mirr = decode.simulation.trafo.pos.trafo.XYZChanneledTransformation(
    t_mirr,
    ch=1,
)
trafo.append(t_mirr)
trafo_shift = decode.simulation.trafo.pos.trafo.XYZShiftTransformation(
    offset, device=device[0]
)
trafo_shift = decode.simulation.trafo.pos.trafo.XYZChanneledTransformation(trafo_shift, 1)

# final trafo
trafo.append(trafo_shift)
trafo = trafo.to("cpu")
trafo

# Infer

In [ ]:
import decode.neuralfitter.inference.functional as infer_func

def patch_load(path_cfg):
    cfg = decode.io.param.load(path_cfg)

    # patches
    cfg["Paths"]["trafo"] = path_trafo
    cfg = decode.io.param.patch_img_size(cfg, img_shape, scenario=["Simulation", "Test"])

    for i in range(2):
        cfg["Camera"][i]["specs"] |= {"flip": {"gain": None, "channel": None}}

    # scoped fixes
    for s in ["Simulation", "Test"]:
        # integer offset
        # cfg[s]["Transformation"]["Pos"]["glob"]["offset"] = {"x": [0, -14], "y": [0, 0], "z": [0, 0]}
        cfg[s]["Transformation"]["Pos"]["glob"]["offset"] = {"x": [0, 0], "y": [0, 0], "z": [0, 0]}

    return cfg

In [ ]:
scen_model["em"] = None

for i, r in scen_model.iterrows():
    if r["path_model"] is None:
        continue

    path_model = Path(r["path_model"])
    path_ckpt = sorted(path_model.glob("*.ckpt"))[0]
    path_cfg = path_model / "param_run.yaml"

    log.info(f"Loading model, config for scenario", index=i, path_model=path_model, path_ckpt=path_ckpt, path_cfg=path_cfg)

    cfg = patch_load(path_cfg)
    for s in cfg["Hardware"]["device"]:
        cfg["Hardware"]["device"][s] = device[0]

    f = frames[i[-1]]

    em_out, logger = infer_func.infer(
        f,
        frame_crop=(256, 256),
        cfg=cfg,
        model=path_ckpt,
        trafo=trafo,
        mode="multi",
        mode_camera="cameras",
        logger="debug",
        batch_size = 4,
        device = device,
    )

    scen_model.at[i, "em"] = em_out

scen_model

In [ ]:
path_em = path_scen_base / "decode_fit.pt"
torch.save(scen_model.droplevel(0)["em"].to_dict(), path_em)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# plot all inputs and raw outputs
f_ix = 2

f, axs = plt.subplots(ncols=4, nrows=3, figsize=(14, 8))
for i, ax in enumerate(axs.flatten()):
    x = logger.model_in[0][f_ix]
    if i >= x.shape[0]:
        ax.axis("off")
        continue
    im = ax.imshow(x[i])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)

plt.tight_layout()
plt.show()

f, axs = plt.subplots(ncols=4, nrows=3, figsize=(14, 8))
for i, ax in enumerate(axs.flatten()):
    x = logger.model_out[0][f_ix].cpu().numpy()
    if i >= x.shape[0]:
        ax.axis("off")
        continue

    im = ax.imshow(x[i])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
plt.tight_layout()
plt.show()